In [ ]:
import numpy as np
import pandas as pd
import price
from scipy import stats
from data_profiling import ProfileReport
from matplotlib import pyplot at plt
import seaborn as sns
%matplotlib inline

In [ ]:
ahs = pd.read_csv('ahs2023.csv')

In [ ]:
ahs.head(3).T

In [ ]:
profile = ProfileReport(ahs,
                        title = 'American Housing Survey 2023',
                        html = {'style': {'full_width': True}},
                        minimal = True)

In [ ]:
profile.to_notebook_iframe()

In [ ]:
profile.to_file('ahs_dashboard.html')

In [ ]:
ahs['TENURE'].value_counts()

In [ ]:
ahs['HINCP'].describe()

In [ ]:
ahs.groupby('TENURE').agg({'HINCP'}:['count', 'mean', 'median'])

In [ ]:
inc_own = ahs.query("TENURE == 'Owned or being bought by someone in your household")['HINCP']
inc_rent= ahs.query("TENURE == 'Rented'")['HINCP']
inc_squat= ahs.query("TENURE == 'Occupied without payment or rent'")['HINCP']

In [ ]:
stats.f_oneway(inc_own, inc_rent, inc_squat)

In [ ]:
ahs.columns

In [ ]:
broken = ahs [['FUSEBLOW','SEWBREAK', 'ROACH', 'RODENT', 'NOWIRE', 'PLUGS', 'COLD',
               'NOTOIL' ,'NOWAT', 'FLOORHOLE', ' FNDCRUMB', 'PAINTPEEL', 'ROOFHOLE', 'ROOFSAG',
               'ROOFSHIN', "WALLCRACK', 'WALLSIDE', 'WALLSLOPE', 'WINBOARD',
               'WINBROKE', 'LEAKI', 'MOLDBATH']].dropna()

In [ ]:
MCA = prince.MCA(n_components=2)
mca_results = MCA.fit(broken)

In [ ]:
mca_results.row_coordinate(broken)

In [ ]:
mca_results.columns_coordinate(broken).sort_values(0)

In [ ]:
ahs_broken = mca_results.row_coordinates(broken)
ahs_broken = ahs_broken.reset_index()

ahs_broken = ahs_broken.rename({0: 'house_disrepair_index',
                                1: 'structure_vs_utilities'}, axis=1)

ahs_broken

In [ ]:
ahs=ahs.reset_index()

ahs=pd.merge(ahs, ahs_broken,
              on='index',
              how='outer',
              validate='one_to_one',
              indicator='matched')
ahs['matched'].value_counts()

In [ ]:
ahs.head(3).T

In [ ]:
sns.histogram(ahs, x='HINCP')

In [ ]:
fig, ax = plt.subplots(figsize = (10,4))
ax.set(xscale='log',
       xlabel='Household Income (log-10)')
sns.hisplot(ahs.query('HINCP > 0'), x='HINCP')

In [ ]:
fig, ax = plt.subplots(figsize = (10,4))
ax.set(xscale='log',
       xlabel='Household Income (log-10)')

sns.violinplot(ahs.query('HINCP > 0'), x="HINCP")

In [ ]:
fig, ax = plt.subplots(figsize = (10,4))
ax.set(xscale='log',
       xlabel='Household Income (log-10)')

sns.kdeplot(ahs.query('HINCP > 0'), x="HINCP")

In [ ]:
ahs['TENURE'].value_count()

In [ ]:
sns.barplot(ahs, x='TENURE')

In [ ]:
ahs_bar = ahs.groupby('TENURE').size().reset_index()
ahs_bar = ahs_bar.rename({0: 'count'}, axis=1)
sns.barplot(ahs_bar, y="TENURE", x='count', hue='TENURE')

In [ ]:
ahs_bar = ahs.groupby('TENURE').agg({'house_disrepair_index':'mean'}).reset_index()

sns.barplot(ahs_bar, y="TENURE", x='house_disrepair_index', hue='TENURE')

In [ ]:
fig, ax = plt.subplots(figsize = (8,4))
ax.set(xscale='log',
       xlabel='Market Value (log-10)')

sns.scatterplot(ahs, x="MARKETVAL", y='house_disrepair_index', alpha=0.1)

In [ ]:
fig, ax = plt.subplots(figsize = (8,4))
ax.set(xscale='log',
       xlabel='Market Value (log-10)')

sns.regplot(ahs, x="MARKETVAL", y='house_disrepair_index', scatter=False)

In [ ]:
ahs['YRBUILD'].value_count()


In [ ]:
ahs_line = ahs.groupby('YRBUILD').agg({'MARKETVAL':'mean'}).reset_index()

sns.lineplot(ahs_line, x='YRBUILD', y='MARKETVAL')

In [ ]:
ahs_facet =ahs.groupby(['DIVISION', "TENURE"]).size().reset_index()
ahs_facet=ahs_facet.rename({0:'count'}, axis=1)
ahs_facet

g = sns.FacetGrid(data=ahs_facet, col='DIVISION' col_wrap=3, hue='DIVISION')
g.map(sns.barplot, 'count', 'TENURE')
g.set_titles("{col_name}")